In [1]:
pip install denoising_diffusion_pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.7/83.7 kB 4.3 MB/s eta 0:00:00


In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import transforms
import torchvision.datasets as datasets
from torchvision.utils import save_image, make_grid
from torch.utils.data import DataLoader
from denoising_diffusion_pytorch import Unet, GaussianDiffusion
import matplotlib.pyplot as plt

In [3]:
# Hyperparameters
LEARNING_RATE = 4e-4
BATCH_SIZE = 128  # Batch size
N_EPOCHS = 70
IMAGE_SIZE = 28
TIME_STEPS = 1000
SAMPLING_TIMESTEPS = 250

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Nytt avsnitt

In [4]:

SAMPLE_EVERY = 5 # sample every N epochs (and at epoch 1)
NUM_SAMPLES = 16 # how many digits to generate each time
SAMPLE_NROW = 9 # grid width


if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

os.makedirs("samples", exist_ok=True)


Using device: cuda


In [5]:
#we define a tranform that converts the image to tensor
myTransforms = transforms.Compose([transforms.ToTensor()])

# the MNIST dataset is available through torchvision.datasets
print("loading MNIST digits dataset")
dataset = datasets.MNIST(root="dataset/", transform=myTransforms, download=True)
# let's create a dataloader to load the data in batches
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

test_dataset = datasets.MNIST(root='dataset/', train=False, download=False, transform=myTransforms)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

loading MNIST digits dataset


100%|██████████| 9.91M/9.91M [00:00<00:00, 18.6MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 502kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.62MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 12.6MB/s]


In [6]:
myTransforms = transforms.Compose([transforms.ToTensor()])

print("loading MNIST digits dataset")
dataset = datasets.MNIST(root="dataset/", transform=myTransforms, download=True)
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,  
    pin_memory=(device.type == "cuda"),
)

test_dataset = datasets.MNIST(root="dataset/", train=False, download=False, transform=myTransforms)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


loading MNIST digits dataset


In [7]:
DIM = 32
DIM_MULTS = (1, 2, 5)
model = Unet(
    dim = DIM,
    dim_mults = DIM_MULTS,
    flash_attn = False,
    channels = 1
)

diffusion = GaussianDiffusion(
    model,
    image_size = IMAGE_SIZE,
    timesteps = TIME_STEPS,           # number of steps
    sampling_timesteps = SAMPLING_TIMESTEPS    # number of sampling timesteps (using ddim for faster inference [see ddim paper])
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)


In [8]:
epoch_losses = [] # to store losses

for epoch in range(N_EPOCHS): # loop over epoch
    diffusion.train()
    running_loss = 0.0

    for images, _ in loader:
        images = images.to(device) #move to gpu
        loss = diffusion(images) #loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(loader)
    epoch_losses.append(avg_loss)
    print(f"Epoch {epoch + 1}/{N_EPOCHS} | loss: {avg_loss:.4f}")

    if (epoch + 1) % SAMPLE_EVERY == 0:
        diffusion.eval()
        with torch.no_grad():
            samples = diffusion.sample(batch_size=NUM_SAMPLES)
        save_image(samples, f"samples/epoch_{epoch + 1:03d}.png", nrow=SAMPLE_NROW)




Epoch 1/70 | loss: 0.1085
Epoch 2/70 | loss: 0.0509
Epoch 3/70 | loss: 0.0463
Epoch 4/70 | loss: 0.0426
Epoch 5/70 | loss: 0.0417


sampling loop time step:   0%|          | 0/250 [00:00<?, ?it/s]

Epoch 6/70 | loss: 0.0404
Epoch 7/70 | loss: 0.0397
Epoch 8/70 | loss: 0.0390
Epoch 9/70 | loss: 0.0386
Epoch 10/70 | loss: 0.0387


sampling loop time step:   0%|          | 0/250 [00:00<?, ?it/s]

Epoch 11/70 | loss: 0.0384
Epoch 12/70 | loss: 0.0383
Epoch 13/70 | loss: 0.0379
Epoch 14/70 | loss: 0.0375
Epoch 15/70 | loss: 0.0373


...

sampling loop time step:   0%|          | 0/250 [00:00<?, ?it/s]

Epoch 61/70 | loss: 0.0348
Epoch 62/70 | loss: 0.0349
Epoch 63/70 | loss: 0.0347
Epoch 64/70 | loss: 0.0346
Epoch 65/70 | loss: 0.0344


sampling loop time step:   0%|          | 0/250 [00:00<?, ?it/s]

Epoch 66/70 | loss: 0.0347
Epoch 67/70 | loss: 0.0347
Epoch 68/70 | loss: 0.0346
Epoch 69/70 | loss: 0.0345
Epoch 70/70 | loss: 0.0344


sampling loop time step:   0%|          | 0/250 [00:00<?, ?it/s]

In [10]:
plt.figure()
plt.plot(epoch_losses)
plt.xlabel("Epoch")
plt.ylabel("Avg. loss")
plt.title("Diffusion training loss")
plt.savefig("loss_curve.png")
plt.close()